In [1]:
# import libraries
import numpy as np
import pickle
from collections import defaultdict
import scipy as sp
import concurrent.futures
import os

In [3]:
def categorize_diffusers(diffuser_coordinates, fg_coordinates, k, max_distance=5):
    """"Returns arrays containing the types and chain indices of the k closest FGs for each diffuser.

    Args:
        diffuser_coordinates (np.array): a coordinates array of shape [n_diffusers, 3]
        fg_coordinates (Dictionary(string : np.array)): a coordinates dictionary of shape {fg_type : [n_chains, n_beads, 3]}
        k (int): Number of closest FGs to find for each diffuser
        
    Returns:
        np.array: Array of shape [n_diffusers, k] containing strings in format "<fg_type>_<chain_i>"
        or "cyt"/"nuc" if no FG is within max_distance.
    """
    n_diffusers = diffuser_coordinates.shape[0]
    # Initialize results array
    result = np.full((n_diffusers, k), "cyt", dtype='U20')
    result[diffuser_coordinates[:,2] < 0, :] = "nuc"
    
    # Pre-calculate total number of chains
    total_chains = sum(coords.shape[0] for coords in fg_coordinates.values())
    
    # Pre-allocate arrays for all chains
    all_min_distances = np.zeros((n_diffusers, total_chains))
    all_formatted_strings = np.empty(total_chains, dtype='U20')
    
    # Calculate minimum distances for all chains at once
    current_idx = 0
    for fg_type, coords in fg_coordinates.items():
        n_chains = coords.shape[0]
        
        # Generate formatted strings for this FG type
        chain_strings = np.array([f"{fg_type}_{i:02d}" for i in range(n_chains)])
        all_formatted_strings[current_idx:current_idx + n_chains] = chain_strings
        
        # Reshape arrays for broadcasting
        chains_reshaped = coords.reshape(n_chains, -1, 3)  # [n_chains, n_beads, 3]
        diffusers_expanded = diffuser_coordinates[:, np.newaxis, np.newaxis, :]  # [n_diffusers, 1, 1, 3]
        
        # Calculate distances to all beads for all chains at once
        distances = np.linalg.norm(chains_reshaped[np.newaxis, :, :, :] - diffusers_expanded, axis=3)
        
        # Find minimum distance per chain
        all_min_distances[:, current_idx:current_idx + n_chains] = np.min(distances, axis=2)
        
        current_idx += n_chains
    
    # Process each diffuser using vectorized operations
    for i in range(n_diffusers):
        # Find valid distances within max_distance
        valid_mask = all_min_distances[i] <= max_distance
        
        if np.any(valid_mask):
            # Get valid distances and their indices
            valid_distances = all_min_distances[i][valid_mask]
            valid_strings = all_formatted_strings[valid_mask]
            
            # Get indices of k smallest distances
            k_smallest_indices = np.argsort(valid_distances)[:k]
            result[i, :len(k_smallest_indices)] = valid_strings[k_smallest_indices]
    
    return result


def categorize_diffusers_over_time(diffuser_trajectories, fg_trajectories, k, step=1):
    n_diffusers = diffuser_trajectories.shape[0]
    n_t = diffuser_trajectories.shape[2]
    n_t2 = fg_trajectories["Nup57"].shape[3]
    if n_t != n_t2:
        raise Exception("Times not matching")
    n_columns = len(range(0, n_t, step))
    categorized_trajectories = np.zeros(shape=(n_diffusers, k, n_columns), dtype='U20')
    for t in range(0, n_t, step):
        cur_diffuser_trajectories = diffuser_trajectories[:, :, t]
        cur_fg_trajectories = {fg_type : fg_trajectories[:, :, :, t] for (fg_type, fg_trajectories) in fg_trajectories.items()}
        categorized_trajectories[:,:,int(t / step)] = categorize_diffusers(cur_diffuser_trajectories, cur_fg_trajectories, k)
    return categorized_trajectories

def normalize_rows(counts_matrix):
    transition_matrix = np.zeros_like(counts_matrix)
    n_states = counts_matrix.shape[0]
    for i in range(n_states):
        row_sum = np.sum(counts_matrix[i, :])
        if row_sum == 0:
            print(f"Row sum is 0, setting row {i} to 0    :( ")
            transition_matrix[i, :] = 0
            continue
        transition_matrix[i, :] = counts_matrix[i, :] / row_sum
    return transition_matrix

def generate_counts_matrix(data, fg_types, n_chains_per_fg):
    # Get unique states
    states = []
    for i, fg_type in enumerate(fg_types):
        for j in range(n_chains_per_fg[i]):
            states.append(f"{fg_type}_{j:02d}")
    states = sorted(states)
    states.append("nuc")
    states.append("cyt")
    n_states = len(states)
    
    # Create state to index mapping
    state_to_idx = {state: idx for idx, state in enumerate(states)}
    
    # Initialize transition counts
    counts = defaultdict(lambda: defaultdict(int))
    
    # Count transitions
    for sequence in data:
        for t in range(len(sequence) - 1):
            current_state = sequence[t]
            next_state = sequence[t + 1]
            counts[current_state][next_state] += 1
    return counts, states

def eightwise_symmetrize(data):
    """Given a matrix, makes it 8-wise symmetric"""
    if (data.shape[0] % 8 != 0) or (data.shape[1] % 8 != 0):
        raise Exception("invalid matrix shape")
    mask_base = np.array(np.eye(8, dtype=bool))
    masks = [np.roll(mask_base, shift=i, axis=0) for i in range(8)]
    new_data = np.zeros_like(data)
    for mask in masks:
        for i in range(int(data.shape[0] / 8)):
            for j in range(int(data.shape[1] / 8)):
                new_data[i*8:(i+1)*8, j*8:(j+1)*8][mask] = np.mean(data[i*8:(i+1)*8, j*8:(j+1)*8][mask])
    return new_data

def eightwise_symmetrize_vec(data):
    if data.ndim != 1:
        raise Exception("not a vector")
    if data.shape[0] % 8 != 0:
        raise Exception("invalid vector shape")
    for i in range(int(data.shape[0] / 8)):
        data[i*8:(i+1)*8] = np.mean(data[i*8:(i+1)*8])
    return data

def generate_transition_matrix(data, fg_types, n_chains_per_fg, init_1 = False, symmetrize = False, add_to_diag = 0):
    """
    Generate a transition matrix from an array of categorical time series.
    
    Parameters:
    -----------
    data : array-like
        Array of shape [n_diffusers, t] containing categorical data (strings)
        
    Returns:
    --------
    transition_matrix : numpy.ndarray
        Matrix of transition probabilities
    states : list
        List of unique states (categories)
    """
    counts, states = generate_counts_matrix(data, fg_types, n_chains_per_fg)
    n_states = len(states)
    
    # Create transition matrix
    if init_1: transition_matrix = np.ones((n_states, n_states))
    else: transition_matrix = np.zeros((n_states, n_states))
    transition_matrix += add_to_diag * np.eye(n_states)
    
    # Fill transition matrix with probabilities
    for i, state_i in enumerate(states):
        for j, state_j in enumerate(states):
            transition_matrix[i, j] += counts[state_i][state_j]
    if symmetrize:
        transition_matrix[:-2, -1] = eightwise_symmetrize_vec(transition_matrix[:-2, -1])
        transition_matrix[:-2, -2] = eightwise_symmetrize_vec(transition_matrix[:-2, -2])
        transition_matrix[-1, :-2] = eightwise_symmetrize_vec(transition_matrix[-1, :-2])
        transition_matrix[-2, :-2] = eightwise_symmetrize_vec(transition_matrix[-2, :-2])
        transition_matrix[:-2, :-2] = eightwise_symmetrize(transition_matrix[:-2, :-2])
    transition_matrix = normalize_rows(transition_matrix)
    
    return transition_matrix, states

In [4]:
def categorize_multiples(sim_indexes, sim_times, load_path_prefix="data/singles/", step=1, save_file_path=None):
    i_time_iterator = [(sim_i, time_i, time) for sim_i in sim_indexes for time_i, time in enumerate(sim_times)]


    arrays = [["temp" for _ in range(len(sim_times))] for _ in range(len(sim_indexes))]    

    def process_file(i_time):
        sim_i, time_i, time = i_time
        print(f"{time}, {sim_i} ", end="")
        with open(f"{load_path_prefix}/{sim_i}/{time}.pickle", "rb") as f:
            diffuser_trajectories = pickle.load(f)
        with open(f"{load_path_prefix}/{sim_i}/{time}-fgs.pickle", "rb") as f:
            fg_trajectories = pickle.load(f)
        categorized = categorize_diffusers_over_time(diffuser_trajectories, fg_trajectories, 1, step=step)
        arrays[sim_i - 1][time_i] = categorized

    num_processes = len(os.sched_getaffinity(0))
    with concurrent.futures.ThreadPoolExecutor(max_workers=num_processes) as executor:
        executor.map(process_file, i_time_iterator)

    for i in range(len(arrays)):
        arrays[i] = np.concatenate(arrays[i], axis=2)
    all = np.concatenate(arrays, axis=0)

    if save_file_path is not None:
        with open(save_file_path, "wb") as f:
            pickle.dump(all[:, 0, :], f)
    return all[:, 0, :]

def generate_transition_matrix_from_categorized(load_categorized_path,
                                                save_file_path,
                                                fg_types=['Nup2', 'Nsp1', 'Nup100', 'Nup116', 'Nup159', 'Nup49', 'Nup57', 'Nup145', 'Nup1', 'Nup60'],
                                                n_chains_per_fg=[16, 48, 16, 16, 16, 32, 32, 16, 8, 16],
                                                init_1=True,
                                                add_to_diag=0):
    with open(load_categorized_path, "rb") as f:
        categorized = pickle.load(f)
    tm, states = generate_transition_matrix(categorized, fg_types, n_chains_per_fg, symmetrize=True, init_1=init_1, add_to_diag=add_to_diag)
    with open(save_file_path, "wb") as f:
        pickle.dump((tm, states), f)

In [ ]:
for step in [1, 2, 4, 8, 16, 32, 64]:
    generate_transition_matrix_from_multiple(
    sim_indexes = range(1, 51),
    sim_times = ["150-160", "160-170", "170-180"],
    file_name = f"data/transition_matrices/interactions-150-180-symmetric-{100*step}ns.pickle",
    step = step
    )

In [ ]:
generate_transition_matrix_from_multiple(
sim_indexes = range(1, 25),
sim_times = ["200-210"],
file_name = f"data/mutations/no-nsp1/transition_matrices/interactions-200-210-symmetric-100ns.pickle",
step = 1,
load_path_prefix="data/mutations/no-nsp1/singles",
fg_types=['Nup2', 'Nup100', 'Nup116', 'Nup159', 'Nup49', 'Nup57', 'Nup145', 'Nup1', 'Nup60'],
n_chains_per_fg=[16, 16, 16, 16, 32, 32, 16, 8, 16]
)

generate_transition_matrix_from_multiple(
sim_indexes = range(1, 25),
sim_times = ["200-210"],
file_name = f"data/mutations/no-nup100/transition_matrices/interactions-200-210-symmetric-100ns.pickle",
step = 1,
load_path_prefix="data/mutations/no-nup100/singles",
fg_types=['Nup2', 'Nsp1', 'Nup116', 'Nup159', 'Nup49', 'Nup57', 'Nup145', 'Nup1', 'Nup60'],
n_chains_per_fg=[16, 48, 16, 16, 32, 32, 16, 8, 16]
)

generate_transition_matrix_from_multiple(
sim_indexes = range(1, 25),
sim_times = ["200-210"],
file_name = f"data/mutations/no-nup116/transition_matrices/interactions-200-210-symmetric-100ns.pickle",
step = 1,
load_path_prefix="data/mutations/no-nup116/singles",
fg_types=['Nup2', 'Nsp1', 'Nup100', 'Nup159', 'Nup49', 'Nup57', 'Nup145', 'Nup1', 'Nup60'],
n_chains_per_fg=[16, 48, 16, 16, 32, 32, 16, 8, 16]
)

200-210, 1 200-210, 2 200-210, 3 200-210, 4 200-210, 5 200-210, 6 200-210, 7 200-210, 8 200-210, 9 200-210, 10 200-210, 11 200-210, 12 200-210, 13 200-210, 14 200-210, 15 200-210, 16 200-210, 17 200-210, 18 200-210, 19 200-210, 20 200-210, 21 200-210, 22 200-210, 23 200-210, 24 200-210, 1 200-210, 2 200-210, 3 200-210, 4 200-210, 5 200-210, 6 200-210, 7 200-210, 8 200-210, 9 200-210, 10 200-210, 11 200-210, 12 200-210, 13 200-210, 14 200-210, 15 200-210, 16 200-210, 17 200-210, 18 200-210, 19 200-210, 20 200-210, 21 200-210, 22 200-210, 23 200-210, 24 200-210, 1 200-210, 2 200-210, 3 200-210, 4 200-210, 5 200-210, 6 200-210, 7 200-210, 8 200-210, 9 200-210, 10 200-210, 11 200-210, 12 200-210, 13 200-210, 14 200-210, 15 200-210, 16 200-210, 17 200-210, 18 200-210, 19 200-210, 20 200-210, 21 200-210, 22 200-210, 23 200-210, 24 

In [ ]:
# def count_transports(sim):
#     sim = sim.tolist()
#     transports = 0
#     # cur = -1
#     # for i in range(len(sim)-1):
#     #     if cur == -1:
#     #         if sim[i+1] == "nuc" or sim[i+1] == "cyt":
#     #             cur = sim[i+1]
#     #             continue
#     #     elif cur == "nuc" and sim[i+1] == "cyt":
#     #         transports += 1
#     #         cur = "cyt"
#     #     elif cur == "cyt" and sim[i+1] == "nuc":
#     #         transports += 1
#     #         cur = "nuc"
#     if "nuc" in sim and "cyt" in sim:
#         transports += 1
#     return transports

# def count_transport_events_multiple( sim_indexes,
#                             sim_times,
#                             step=1,
#                             load_path_prefix="data/singles/"):
#     #############
#     # Load data #
#     #############
#     i_time_iterator = [(i, time) for i in sim_indexes for time in sim_times]

#     def process_file(i_time):
#         i, time = i_time
#         print(f"{time}, {i} ", end="")
#         with open(f"{load_path_prefix}/{i}/{time}.pickle", "rb") as f:
#             diffuser_trajectories = pickle.load(f)
#         with open(f"{load_path_prefix}/{i}/{time}-fgs.pickle", "rb") as f:
#             fg_trajectories = pickle.load(f)
#         return categorize_diffusers_over_time(diffuser_trajectories, fg_trajectories, 1, step=step)

#     arrays = []
#     num_processes = len(os.sched_getaffinity(0))
#     with concurrent.futures.ThreadPoolExecutor(max_workers=num_processes) as executor:
#         results = list(executor.map(process_file, i_time_iterator))

#     arrays.extend(results)
#     all = np.concatenate(arrays, axis=0)
    
#     transports = 0
#     for i in range(all.shape[0]):
#         transports += count_transports(all[i,0,:])
#     print(transports)
#     return transports

In [5]:
count_transport_events_multiple(
sim_indexes = range(30, 40),
sim_times = ["170-180"],
step = 1,
load_path_prefix="data/singles"
)

NameError: name 'count_transport_events_multiple' is not defined

In [6]:
# save just the categorized trajectories

for step in [1]:
    categorize_multiples(sim_indexes=range(1, 51),
                         sim_times=["150-160", "160-170", "170-180"],
                         step=1,
                         save_file_path=f"data/categorized/interactions-150-180-100ns.pickle")

150-160, 1 160-170, 1 170-180, 1 150-160, 2 160-170, 2 170-180, 2 150-160, 3 160-170, 3 170-180, 3 150-160, 4 160-170, 4 170-180, 4 150-160, 5 160-170, 5 170-180, 5 150-160, 6 160-170, 6 170-180, 6 150-160, 7 160-170, 7 170-180, 7 150-160, 8 160-170, 8 170-180, 8 150-160, 9 160-170, 9 170-180, 9 150-160, 10 160-170, 10 170-180, 10 150-160, 11 160-170, 11 170-180, 11 150-160, 12 160-170, 12 170-180, 12 150-160, 13 160-170, 13 170-180, 13 150-160, 14 160-170, 14 170-180, 14 150-160, 15 160-170, 15 170-180, 15 150-160, 16 160-170, 16 170-180, 16 150-160, 17 160-170, 17 170-180, 17 150-160, 18 160-170, 18 170-180, 18 150-160, 19 160-170, 19 170-180, 19 150-160, 20 160-170, 20 170-180, 20 150-160, 21 160-170, 21 170-180, 21 150-160, 22 160-170, 22 170-180, 22 150-160, 23 160-170, 23 170-180, 23 150-160, 24 160-170, 24 170-180, 24 150-160, 25 160-170, 25 170-180, 25 150-160, 26 160-170, 26 170-180, 26 150-160, 27 160-170, 27 170-180, 27 150-160, 28 160-170, 28 170-180, 28 150-160, 29 160-170

In [ ]:
# generate matrix without init_1
for step in [1]:
    generate_transition_matrix_from_multiple(
    sim_indexes = range(1, 51),
    sim_times = ["150-160", "160-170", "170-180"],
    file_name = f"data/transition_matrices/interactions-150-180-symmetric-{100*step}ns-0init.pickle",
    step = step,
    init_1=False
    )

150-160, 1 160-170, 1 170-180, 1 150-160, 2 160-170, 2 170-180, 2 150-160, 3 160-170, 3 170-180, 3 150-160, 4 160-170, 4 170-180, 4 150-160, 5 160-170, 5 170-180, 5 150-160, 6 160-170, 6 170-180, 6 150-160, 7 160-170, 7 170-180, 7 150-160, 8 160-170, 8 170-180, 8 150-160, 9 160-170, 9 170-180, 9 150-160, 10 160-170, 10 170-180, 10 150-160, 11 160-170, 11 170-180, 11 150-160, 12 160-170, 12 170-180, 12 150-160, 13 160-170, 13 170-180, 13 150-160, 14 160-170, 14 170-180, 14 150-160, 15 160-170, 15 170-180, 15 150-160, 16 160-170, 16 170-180, 16 150-160, 17 160-170, 17 170-180, 17 150-160, 18 160-170, 18 170-180, 18 150-160, 19 160-170, 19 170-180, 19 150-160, 20 160-170, 20 170-180, 20 150-160, 21 160-170, 21 170-180, 21 150-160, 22 160-170, 22 170-180, 22 150-160, 23 160-170, 23 170-180, 23 150-160, 24 160-170, 24 170-180, 24 150-160, 25 160-170, 25 170-180, 25 150-160, 26 160-170, 26 170-180, 26 150-160, 27 160-170, 27 170-180, 27 150-160, 28 160-170, 28 170-180, 28 150-160, 29 160-170

In [8]:
# diag pseudo counts
# for sim_i in [0, 1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]:
for sim_i in [2048, 4096, 8192, 16384, 32768, 65536]:
    generate_transition_matrix_from_categorized(load_categorized_path="/cs/usr/roi.eliasian/LabFolder/Master/NPC-markov/data/categorized/interactions-150-180-100ns.pickle",
                                                save_file_path=f"/cs/usr/roi.eliasian/LabFolder/Master/NPC-markov/data/transition_matrices/interactions-150-180-sym-diag+{sim_i}.pickle",
                                                fg_types=['Nup2', 'Nsp1', 'Nup100', 'Nup116', 'Nup159', 'Nup49', 'Nup57', 'Nup145', 'Nup1', 'Nup60'],
                                                n_chains_per_fg=[16, 48, 16, 16, 16, 32, 32, 16, 8, 16],
                                                init_1=True,
                                                add_to_diag=sim_i)